# Phase 9 — Fixed Subset Validation + Window Ablation + QA Domain Transfer

**Goal:** Validate the pre-selected 4-feature subset (`sw_var_peak + trace_length + spectral_centroid + stft_max_high_power`) on new domains (TriviaQA, WebQ) without re-running exhaustive subset search.

**Model:** `tiiuae/Falcon3-10B-Instruct` — same model used in EPR paper baselines for TriviaQA/WebQ.

**Grading:** Normalized exact-match against gold aliases (EPR paper standard; no LLM judge needed).

**Sections:**
1. Setup + Config
2. TriviaQA inference (with checkpointing)
3. WebQ inference (with checkpointing)
4. Feature extraction
5. Feature behavior (distributions by correctness)
6. Correlation heatmap (fixed subset decorrelation check)
7. Window ablation (`sw_var_peak` AUC vs w + adaptive)
8. Fixed subset Nadler fusion (no re-search)
9. Baseline comparison + summary table

In [ ]:
import os, sys, shutil

REPO_DIR = '/content/hallucination_detection'

# If a previous (wrong) clone exists, remove it so we get a clean clone
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    print(f'Removing stale clone at {REPO_DIR} (spectral_utils missing)...')
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    # -b master: our code lives on master; GitHub default branch may differ
    ret = os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
    print('clone exit code:', ret)
else:
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('spectral_utils present:', os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')))

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_with_window, sw_var_peak_adaptive,
    FEAT_NAMES, load_cache, save_cache,
    boot_auc, nadler_fuse, simple_average_fusion, zscore,
)
from spectral_utils.data_loaders import (
    load_trivia_qa, trivia_qa_prompt, is_correct_trivia_qa,
    load_webq, webq_prompt, is_correct_webq,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

print('spectral_utils imported OK')

In [ ]:
# Cell 2 — Config
MODEL_ID    = 'tiiuae/Falcon3-10B-Instruct'
N_SAMPLES   = 300
TEMPERATURE = 1.0
MAX_TOKENS  = 64       # short direct answers — no CoT
CACHE_DIR   = '/content/drive/MyDrive/spectral_phase9_cache'

# Pre-selected 4-feature subset (Phase 5 core stable set)
FIXED_SUBSET = ['sw_var_peak', 'trace_length', 'spectral_centroid', 'stft_max_high_power']

# Window sizes for ablation
WINDOW_SIZES = [3, 5, 7, 9, 12, 16, 24, 32]

print(f'Model  : {MODEL_ID}')
print(f'Subset : {FIXED_SUBSET}')
print(f'Windows: {WINDOW_SIZES}')

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir ready: {CACHE_DIR}')

In [ ]:
# Cell 4 — Load model
# Falcon-3-10B fits on A100 at bfloat16 — no quantization needed
mdl, tok = load_model(MODEL_ID, quantize_4bit=False)

In [ ]:
# Cell 5 — TriviaQA inference (with checkpointing)
TRIVIA_CACHE = f'{CACHE_DIR}/trivia_qa_traces.pkl'

trivia_items   = load_trivia_qa(n_samples=N_SAMPLES)
trivia_results = load_cache(TRIVIA_CACHE) or []
start_idx      = len(trivia_results)
print(f'TriviaQA: resuming from {start_idx}/{N_SAMPLES}')

for i, item in enumerate(trivia_items[start_idx:], start=start_idx):
    prompt = trivia_qa_prompt(item['question'])
    text, ents = generate_full(mdl, tok, prompt,
                               temperature=TEMPERATURE, max_new_tokens=MAX_TOKENS)
    correct = is_correct_trivia_qa(text, item)
    trivia_results.append({'text': text, 'ents': ents, 'correct': correct, 'item': item})
    if (i + 1) % 25 == 0:
        save_cache(trivia_results, TRIVIA_CACHE)
        acc_so_far = np.mean([r['correct'] for r in trivia_results])
        print(f'  {i+1}/{N_SAMPLES}  acc={acc_so_far:.1%}')

save_cache(trivia_results, TRIVIA_CACHE)
trivia_acc = np.mean([r['correct'] for r in trivia_results])
n_correct  = sum(r['correct'] for r in trivia_results)
print(f'\nTriviaQA done. Acc={trivia_acc:.1%}  ({n_correct}/{len(trivia_results)} correct)')

In [ ]:
# Cell 6 — WebQ inference (with checkpointing)
WEBQ_CACHE = f'{CACHE_DIR}/webq_traces.pkl'

webq_items   = load_webq(n_samples=N_SAMPLES)
webq_results = load_cache(WEBQ_CACHE) or []
start_idx    = len(webq_results)
print(f'WebQ: resuming from {start_idx}/{N_SAMPLES}')

for i, item in enumerate(webq_items[start_idx:], start=start_idx):
    prompt = webq_prompt(item['question'])
    text, ents = generate_full(mdl, tok, prompt,
                               temperature=TEMPERATURE, max_new_tokens=MAX_TOKENS)
    correct = is_correct_webq(text, item)
    webq_results.append({'text': text, 'ents': ents, 'correct': correct, 'item': item})
    if (i + 1) % 25 == 0:
        save_cache(webq_results, WEBQ_CACHE)
        acc_so_far = np.mean([r['correct'] for r in webq_results])
        print(f'  {i+1}/{N_SAMPLES}  acc={acc_so_far:.1%}')

save_cache(webq_results, WEBQ_CACHE)
webq_acc  = np.mean([r['correct'] for r in webq_results])
n_correct = sum(r['correct'] for r in webq_results)
print(f'\nWebQ done. Acc={webq_acc:.1%}  ({n_correct}/{len(webq_results)} correct)')

In [ ]:
# Cell 7 — Unload model to free GPU memory for analysis
del mdl, tok
free_memory()
print('Model unloaded.')

In [ ]:
# Cell 8 — Feature extraction
def extract_dataset_features(results):
    """Extract all 12 spectral features + adaptive sw_var_peak."""
    rows, labels = [], []
    skipped = 0
    for r in results:
        feats = extract_all_features(r['ents'])
        if feats is None:
            skipped += 1
            continue
        feats['sw_var_peak_adaptive'] = sw_var_peak_adaptive(r['ents'])
        rows.append(feats)
        labels.append(int(r['correct']))
    if skipped:
        print(f'  Skipped {skipped} traces (too short for spectral analysis)')
    return pd.DataFrame(rows), np.array(labels)

trivia_df, trivia_labels = extract_dataset_features(trivia_results)
webq_df,   webq_labels   = extract_dataset_features(webq_results)

print(f'TriviaQA: {len(trivia_df)} samples, {trivia_labels.mean():.1%} correct')
print(f'WebQ:     {len(webq_df)} samples, {webq_labels.mean():.1%} correct')
print(f'\nAll features: {list(trivia_df.columns)}')

## Feature Behavior

Distribution of each fixed-subset feature split by correctness.
We expect `sw_var_peak` to show clear separation (lower variance peak → more confident → more likely correct).
`trace_length` is expected to correlate weakly — short QA answers don't vary much in length.

In [ ]:
# Cell 9 — Feature behavior: distributions by correctness
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Feature Distributions: Correct vs Hallucination\n(Falcon-3-10B)', fontsize=13)

for col_idx, feat in enumerate(FIXED_SUBSET):
    for row_idx, (df, labels, dname) in enumerate([
        (trivia_df, trivia_labels, 'TriviaQA'),
        (webq_df,   webq_labels,   'WebQ'),
    ]):
        ax = axes[row_idx, col_idx]
        correct_vals = df[feat][labels == 1].values
        wrong_vals   = df[feat][labels == 0].values
        ax.hist(correct_vals, bins=30, alpha=0.6, color='steelblue', label='Correct', density=True)
        ax.hist(wrong_vals,   bins=30, alpha=0.6, color='tomato',    label='Wrong',   density=True)
        auc, lo, hi = boot_auc(labels, df[feat].values)
        ax.set_title(f'{feat}\n{dname} — AUC {100*auc:.1f}%  [{100*lo:.0f}, {100*hi:.0f}]',
                     fontsize=8.5)
        if col_idx == 0:
            ax.set_ylabel(dname, fontsize=10)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{CACHE_DIR}/feature_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation Heatmap

We validate that the 4 selected features remain decorrelated on new domains.
Nadler fusion works best when pairwise Spearman |ρ| < 0.75 for all pairs in the subset.

In [ ]:
# Cell 10 — Spearman correlation heatmap for fixed subset
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Spearman Correlation — Fixed Subset Features', fontsize=12)

for ax, (df, dname) in zip(axes, [(trivia_df, 'TriviaQA'), (webq_df, 'WebQ')]):
    n = len(FIXED_SUBSET)
    corr_mat = np.zeros((n, n))
    for i, f1 in enumerate(FIXED_SUBSET):
        for j, f2 in enumerate(FIXED_SUBSET):
            r, _ = spearmanr(df[f1].values, df[f2].values)
            corr_mat[i, j] = r
    sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
                xticklabels=FIXED_SUBSET, yticklabels=FIXED_SUBSET, ax=ax,
                annot_kws={'size': 9})
    ax.set_title(dname, fontsize=11)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
    # Annotate pairs that exceed the Nadler threshold
    for i in range(n):
        for j in range(n):
            if i != j and abs(corr_mat[i, j]) >= 0.75:
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='black', lw=2))

plt.tight_layout()
plt.savefig(f'{CACHE_DIR}/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Black borders = |rho| >= 0.75 (Nadler threshold).')

## Window Ablation

We test `sw_var_peak` AUC across a range of window sizes for short QA traces,
plus the adaptive window (≈10% of trace length, clipped to [3, 32]).

For math traces (~1000 tokens), w=16 was optimal. For QA traces (~50–100 tokens),
w=16 covers 16–32% of the trace — potentially too coarse.

In [ ]:
# Cell 11 — Window ablation: compute AUC for each window size + adaptive
def window_ablation(results, labels, window_sizes, dataset_name):
    aucs = {}
    for w in window_sizes:
        scores = np.array([sw_var_peak_with_window(r['ents'], w) for r in results])
        auc, lo, hi = boot_auc(labels, scores)
        aucs[w] = (auc, lo, hi)
        print(f'  {dataset_name}  w={w:2d}: AUC={100*auc:.1f}%  CI=[{100*lo:.1f}, {100*hi:.1f}]')
    adaptive_scores = np.array([sw_var_peak_adaptive(r['ents']) for r in results])
    auc_a, lo_a, hi_a = boot_auc(labels, adaptive_scores)
    # Report the actual window used (median trace length)
    trace_lens = [len(r['ents']) for r in results]
    median_w   = max(3, min(32, int(np.median(trace_lens) * 0.10)))
    print(f'  {dataset_name}  adaptive (median_w≈{median_w}): '
          f'AUC={100*auc_a:.1f}%  CI=[{100*lo_a:.1f}, {100*hi_a:.1f}]')
    return aucs, (auc_a, lo_a, hi_a), median_w

print('=== TriviaQA ===')
trivia_window_aucs, trivia_adaptive, trivia_median_w = window_ablation(
    trivia_results, trivia_labels, WINDOW_SIZES, 'TriviaQA')

print('\n=== WebQ ===')
webq_window_aucs, webq_adaptive, webq_median_w = window_ablation(
    webq_results, webq_labels, WINDOW_SIZES, 'WebQ')

In [ ]:
# Cell 12 — Plot window ablation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('sw_var_peak AUC vs Window Size (Falcon-3-10B)', fontsize=12)

for ax, (window_aucs, adaptive, median_w, dname) in zip(axes, [
    (trivia_window_aucs, trivia_adaptive, trivia_median_w, 'TriviaQA'),
    (webq_window_aucs,   webq_adaptive,   webq_median_w,   'WebQ'),
]):
    ws   = list(window_aucs.keys())
    aucs = [window_aucs[w][0] * 100 for w in ws]
    los  = [window_aucs[w][1] * 100 for w in ws]
    his  = [window_aucs[w][2] * 100 for w in ws]

    ax.plot(ws, aucs, 'o-', color='steelblue', label='Fixed window', zorder=3)
    ax.fill_between(ws, los, his, alpha=0.2, color='steelblue')

    auc_a = adaptive[0] * 100
    lo_a  = adaptive[1] * 100
    hi_a  = adaptive[2] * 100
    ax.axhline(auc_a, color='tomato', linestyle='--',
               label=f'Adaptive (≈w={median_w}) — {auc_a:.1f}%')
    ax.fill_between([ws[0], ws[-1]], [lo_a, lo_a], [hi_a, hi_a],
                    alpha=0.15, color='tomato')

    ax.set_xlabel('Window size w', fontsize=11)
    ax.set_ylabel('AUC (%)', fontsize=11)
    ax.set_title(dname, fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xticks(ws)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(f'{CACHE_DIR}/window_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## Fixed Subset Nadler Fusion

Apply the pre-selected 4-feature subset using Nadler fusion — **no new subset search**.
Sign orientation and z-scoring are applied before fusion (same as `best_nadler_on` internally).

In [ ]:
# Cell 13 — Fixed subset fusion
def apply_fixed_subset(df, labels, subset, label=''):
    """Orient, z-score, and Nadler-fuse a pre-defined feature subset. No search."""
    feats_dict = {f: df[f].values for f in subset}

    # Orient: higher score → more likely correct
    signs = {}
    for f in subset:
        a_pos, *_ = boot_auc(labels,  feats_dict[f])
        a_neg, *_ = boot_auc(labels, -feats_dict[f])
        signs[f] = +1 if a_pos >= a_neg else -1

    oriented = {f: zscore(feats_dict[f] * signs[f]) for f in subset}

    fused_nadler, weights = nadler_fuse(*[oriented[f] for f in subset])
    auc_n, lo_n, hi_n    = boot_auc(labels, fused_nadler)

    fused_mean, _ = simple_average_fusion(*[oriented[f] for f in subset])
    auc_m, lo_m, hi_m = boot_auc(labels, fused_mean)

    print(f'\n=== Fixed Subset Fusion [{label}] ===')
    print(f'Subset: {" + ".join(subset)}\n')
    for f, w, s in zip(subset, weights, [signs[f] for f in subset]):
        a, *_ = boot_auc(labels, feats_dict[f])
        print(f'  {f:<30}  indiv AUC={100*a:.1f}%  Nadler weight={w:.3f}  sign={s:+d}')
    print(f'\n  Nadler  : {100*auc_n:.1f}%  [{100*lo_n:.1f}, {100*hi_n:.1f}]')
    print(f'  Mean    : {100*auc_m:.1f}%  [{100*lo_m:.1f}, {100*hi_m:.1f}]')
    print(f'  Lift    : {(auc_n - auc_m)*100:+.1f} pp')
    return auc_n, lo_n, hi_n, auc_m, lo_m, hi_m

trivia_nadler_auc, trivia_nadler_lo, trivia_nadler_hi, trivia_mean_auc, _, _ = apply_fixed_subset(
    trivia_df, trivia_labels, FIXED_SUBSET, label='TriviaQA')

webq_nadler_auc, webq_nadler_lo, webq_nadler_hi, webq_mean_auc, _, _ = apply_fixed_subset(
    webq_df, webq_labels, FIXED_SUBSET, label='WebQ')

## Baseline Comparison and Summary

We compare the fixed subset fusion against:
- **EPR baseline** (`epr` feature = mean token entropy) — the standard single-feature EPR score
- **Individual features** from the fixed subset
- **Adaptive `sw_var_peak`** alone

Reference numbers from prior work (Falcon-3-10B, `direct_fresh`, `Unified_EPR_Ensemble_res.ipynb`):
- TriviaQA EPR: **72.0%**
- WebQ EPR: **66.4%**

In [ ]:
# Cell 14 — Baseline comparison table
rows = []

signals = [
    ('epr',                 'EPR (mean entropy)'),
    ('sw_var_peak',         'sw_var_peak  w=16'),
    ('trace_length',        'trace_length'),
    ('spectral_centroid',   'spectral_centroid'),
    ('stft_max_high_power', 'stft_max_high_power'),
]

for feat, label in signals:
    ta, tlo, thi = boot_auc(trivia_labels, trivia_df[feat].values)
    wa, wlo, whi = boot_auc(webq_labels,   webq_df[feat].values)
    rows.append({'Signal': label,
                 'TriviaQA AUC': f'{100*ta:.1f}%',
                 'WebQ AUC':     f'{100*wa:.1f}%'})

# Adaptive sw_var_peak
ta_a, *_ = boot_auc(trivia_labels,
                    np.array([sw_var_peak_adaptive(r['ents']) for r in trivia_results]))
wa_a, *_ = boot_auc(webq_labels,
                    np.array([sw_var_peak_adaptive(r['ents']) for r in webq_results]))
rows.append({'Signal': 'sw_var_peak  adaptive',
             'TriviaQA AUC': f'{100*ta_a:.1f}%',
             'WebQ AUC':     f'{100*wa_a:.1f}%'})

rows.append({'Signal': '--- fusion ---', 'TriviaQA AUC': '', 'WebQ AUC': ''})
rows.append({'Signal': 'Fixed 4-feat Nadler',
             'TriviaQA AUC': f'{100*trivia_nadler_auc:.1f}%  [{100*trivia_nadler_lo:.0f}, {100*trivia_nadler_hi:.0f}]',
             'WebQ AUC':     f'{100*webq_nadler_auc:.1f}%  [{100*webq_nadler_lo:.0f}, {100*webq_nadler_hi:.0f}]'})
rows.append({'Signal': 'Fixed 4-feat Mean',
             'TriviaQA AUC': f'{100*trivia_mean_auc:.1f}%',
             'WebQ AUC':     f'{100*webq_mean_auc:.1f}%'})

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

epr_trivia, *_ = boot_auc(trivia_labels, trivia_df['epr'].values)
epr_webq,   *_ = boot_auc(webq_labels,   webq_df['epr'].values)
print(f'\nNadler lift over EPR baseline:')
print(f'  TriviaQA: {(trivia_nadler_auc - epr_trivia)*100:+.1f} pp  '
      f'(ref EPR direct_fresh = 72.0% from Unified_EPR_Ensemble_res)')
print(f'  WebQ:     {(webq_nadler_auc - epr_webq)*100:+.1f} pp  '
      f'(ref EPR direct_fresh = 66.4% from Unified_EPR_Ensemble_res)')

In [ ]:
# Cell 15 — Bar chart: fixed subset vs EPR baseline
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Spectral Fixed Subset vs EPR Baseline (Falcon-3-10B)', fontsize=12)

bar_labels = ['EPR\n(mean entropy)', 'sw_var_peak\nw=16', 'sw_var_peak\nadaptive',
              'Fixed 4-feat\nMean fusion', 'Fixed 4-feat\nNadler fusion']

for ax, (df, labels, results, nadler_a, mean_a, dname) in zip(axes, [
    (trivia_df, trivia_labels, trivia_results, trivia_nadler_auc, trivia_mean_auc, 'TriviaQA'),
    (webq_df,   webq_labels,   webq_results,   webq_nadler_auc,   webq_mean_auc,   'WebQ'),
]):
    epr_a,  *_ = boot_auc(labels, df['epr'].values)
    sw_a,   *_ = boot_auc(labels, df['sw_var_peak'].values)
    adap_a, *_ = boot_auc(labels,
                           np.array([sw_var_peak_adaptive(r['ents']) for r in results]))
    aucs   = [epr_a * 100, sw_a * 100, adap_a * 100, mean_a * 100, nadler_a * 100]
    colors = ['#aaaaaa', '#6baed6', '#3182bd', '#fdae6b', '#e6550d']
    bars   = ax.bar(bar_labels, aucs, color=colors, edgecolor='black', linewidth=0.7)
    ax.set_ylim(max(0, min(aucs) - 5), min(100, max(aucs) + 5))
    ax.set_ylabel('AUC (%)', fontsize=11)
    ax.set_title(dname, fontsize=11)
    ax.tick_params(axis='x', labelsize=8)
    for bar, val in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{CACHE_DIR}/summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()